### Recommender systems, example 1, cosine similarity (movies)

In [144]:
# basic imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import seaborn as sns

# Cosines Similarity / Vectorization modules
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [145]:
# load data
df = pd.read_csv("tmdb_5000_movies.csv")
df.head(5)

# NOTE!: many of the fields are JSON-colletions, they are useless scramble of text
# UNLESS THE JSON IS PROCESSED SOMEHOW

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


In [146]:
# let's study the DataFrame.... what is newest movie in this dataset?
df.sort_values(by="release_date", ascending=False).head(3)

# newest movies are from 2016-2017 => you can get a newer version of TMDB or MovieLens
# -datasets if you want to have more new movies in the data

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
4255,0,"[{""id"": 35, ""name"": ""Comedy""}, {""id"": 10751, ""...",http://www.growingupsmithmovie.com,426469,[],en,Growing Up Smith,"In 1979, an Indian family moves to America wit...",0.710870,[],[],2017-02-03,0,102.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,It’s better to stand out than to fit in.,Growing Up Smith,7.4,7
3408,0,"[{""id"": 18, ""name"": ""Drama""}, {""id"": 10749, ""n...",NaN,325373,"[{""id"": 1415, ""name"": ""small town""}, {""id"": 46...",en,Two Lovers and a Bear,Set in a small town near the North Pole where ...,3.139294,[],[],2016-10-02,0,96.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,NaN,Two Lovers and a Bear,6.8,27
3302,8000000,"[{""id"": 18, ""name"": ""Drama""}]",NaN,374461,"[{""id"": 1650, ""name"": ""cook""}, {""id"": 6054, ""n...",en,Mr. Church,A unique friendship develops when a little gir...,7.828459,"[{""name"": ""Envision Media Arts"", ""id"": 19456},...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2016-09-16,0,104.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,He was the one person she could always count on.,Mr. Church,7.0,129


**Pre-processing the dataset (remove unneeded columns, PROCESS THE JSON CORRECTLY)**

In [147]:
# helper function to handle the JSON of one column
def parse_json_or_list(value):
    try:
        return json.loads(value)
    except (TypeError, ValueError):
        # if we can't successfully convert to JSON
        # return the error message regarding the data in the JSON
        return eval(value)

In [148]:
# example of the genre
# basically we need to grab only the names from each genre
# => place it in one string (same logic also for keywords)
# [
    # {'id': 28, 'name': 'Action'}, 
    # {'id': 12, 'name': 'Adventure'}, 
    # {'id': 14, 'name': 'Fantasy'}, 
    # {'id': 878, 'name': 'Science Fiction'}
# ]

# example keywords
# [
    # {'id': 1463, 'name': 'culture clash'}, 
    # {'id': 2964, 'name': 'future'}, 
    # {'id': 3386, 'name': 'space war'}, 
    # {'id': 3388, 'name': 'space colony'}, 
    # {'id': 3679, 'name': 'society'}, 
    # {'id': 3801, 'name': 'space travel'}, 
    # {'id': 9685, 'name': 'futuristic'}, 
    # {'id': 9840, 'name': 'romance'}, 
    # {'id': 9882, 'name': 'space'}, 
    # {'id': 9951, 'name': 'alien'}, 
    # {'id': 10148, 'name': 'tribe'}, 
    # {'id': 10158, 'name': 'alien planet'}, 
    # {'id': 10987, 'name': 'cgi'}, 
    # {'id': 11399, 'name': 'marine'}, 
    # {'id': 13065, 'name': 'soldier'}, 
    # {'id': 14643, 'name': 'battle'}, 
    # {'id': 14720, 'name': 'love affair'}, 
    # {'id': 165431, 'name': 'anti war'}, 
    # {'id': 193554, 'name': 'power relations'}, 
    # {'id': 206690, 'name': 'mind and soul'}, 
    # {'id': 209714, 'name': '3d'}
# ]

In [149]:
# APPLY THE FIXES for genres and keywords
# this converts the original TEXT JSON => JSON OBJECTS
df['genres'] = df['genres'].apply(parse_json_or_list)
df['keywords'] = df['keywords'].apply(parse_json_or_list)

In [150]:
# next idea: combine genres + keywords into one STRING
def genres_and_keywords_to_string(row):
    # get genres and keywords for this row (movie)
    genres = row['genres']
    keywords = row['keywords']

    # join the genres list into a single string (also keywords)
    genres_str = ' '.join(''.join(j['name'].split()) for j in genres)
    keywords_str = ' '.join(''.join(j['name'].split()) for j in keywords)

    return f"{genres_str} {keywords_str}"

In [151]:
# try COMBINING genres + keywords
df['string'] = df.apply(genres_and_keywords_to_string, axis=1)

In [152]:
# example of one row, column string:
# 'Action Adventure Fantasy ScienceFiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d'
# df.iloc[0]['string']

### TfidVectorizer + Cosine Similarity

In [153]:
# initialize TF-IDF -vectorizer with a specified number of features
tfidf = TfidfVectorizer(max_features=2000)

In [154]:
# fit the vectorizer
X = tfidf.fit_transform(df['string'])

In [155]:
# next, we need to map each title to an index
title_to_index = pd.Series(df.index, index=df['title'])

### Create the recommendation function

In [156]:
# define a function the RECOMMENDS movies based on our INPUT movie
def recommend_movies(title):
    # get the index if of the input movie title
    movie_id = title_to_index[title]

    # get the matching TF-IDF -vector for the input movie
    query = X[movie_id]

    # calculate cosine similarities between our INPUT MOVIE
    # and ALL MOVIES in the dataset
    # query = the specific vetor of INPUT movie
    # X = vectors of all movies
    scores = cosine_similarity(query, X)
    
    # remove the nesting from the scores
    scores = scores.flatten()

    # NOTE! the first movie is always the same movie as input
    # because COSINE SIMILARITY is 100% every time
    recommended_movie_id = (-scores).argsort()[1:11]

    # Create a list of tuples (movie title, score) by using list comprehension
    recommendations = [(df['title'].iloc[i], scores[i]) for i in recommended_movie_id]

    return recommendations



In [157]:
# a very crude way to find the movie we are looking for to try this model
# the search should probably also turn everything to lowercase for easier results
search_term = "Amadeus"
print("Partially matching movie titles:\n")
print("\n".join(df[df['original_title'].str.contains(search_term)]['title'].values))

# copy-paste the movie name you wish search down below

Partially matching movie titles:

Amadeus


### This is the final "testing ground" to get some recommendations

In [158]:
# test the recommendation function

# paste the actual name of the movie from the tool above here
input_movie_title = 'Amadeus'
recommended_movie_titles = recommend_movies(input_movie_title)

# let's print the results
print(f"\nRecommendations for '{input_movie_title}':")

for movie_tuple in recommended_movie_titles:
    # unpack the tuple to title and Cosine Similarity -score
    movie_name = movie_tuple[0]
    movie_score = movie_tuple[1]

    # get the row in DataFrame for this movie
    movie_item = df[df['title'] == movie_name].iloc[0]
    popularity = movie_item['popularity']
    vote_average = movie_item['vote_average']

    # in theory, cosine similary functions between -1 and +1 
    # technically -1 = the complete opposite "similarity"
    # in text-based vectorized applications, the range, however
    # is between 0 and 1, 0 implying no similarity at all, +1 perfect similarity (not practically possible)
    # heavy similarities are probably 0.3+
    print(f"- {movie_name} (Popularity: {popularity}, Vote average: {vote_average:.2f}, Similarity: {movie_score:.2f})")


Recommendations for 'Amadeus':
- Beyond the Lights (Popularity: 8.079046, Vote average: 7.00, Similarity: 0.39)
- Mozart's Sister (Popularity: 0.774442, Vote average: 6.20, Similarity: 0.38)
- Walter (Popularity: 2.659006, Vote average: 5.30, Similarity: 0.32)
- The Last of the Mohicans (Popularity: 37.776566, Vote average: 7.10, Similarity: 0.26)
- Cadillac Records (Popularity: 13.427427, Vote average: 6.90, Similarity: 0.26)
- Glitter (Popularity: 3.809131, Vote average: 3.40, Similarity: 0.26)
- Rob Roy (Popularity: 10.703456, Vote average: 6.50, Similarity: 0.24)
- Noah (Popularity: 46.115758, Vote average: 5.60, Similarity: 0.24)
- What's Love Got to Do with It (Popularity: 7.921311, Vote average: 6.90, Similarity: 0.22)
- Shattered Glass (Popularity: 6.27257, Vote average: 6.60, Similarity: 0.22)
